In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import itertools

In [2]:
df = pd.read_csv("claude_output.csv")
df["HCV_active"] = df["HCV"].isin(["acute", "chronic"]).astype(int)

In [7]:
measures = [
    "Withdrawal", "Public_prop", "Recept_sharing", "Partners",
    "Drug_out", "Partner_HCV_prop", "Treatment",
    "Injections", "Alt_route", "HCV_active"
]

In [ ]:
def paired_bootstrap(df, outcome_col, agent_col="Agent", condition_col="Homeless", n_boot=1000, seed=42):
    np.random.seed(seed)

    wide = df.pivot(index=agent_col,
                    columns=condition_col,
                    values=outcome_col).dropna()

    agents = wide.index.values
    diffs = []

    for _ in range(n_boot):
        sampled_agents = np.random.choice(agents, size=len(agents), replace=True)
        sample = wide.loc[sampled_agents]
        diffs.append((sample[1] - sample[0]).mean())

    diffs = np.array(diffs)
    point_est = (wide[1] - wide[0]).mean() #original (non-bootstrap) estimate

    return {
        "estimate": point_est,
        "ci_lower": np.percentile(diffs, 2.5),
        "ci_upper": np.percentile(diffs, 97.5),
        "bootstrap_sd": diffs.std(ddof=1)
    }

In [25]:
results = []

for v in measures:
    result = paired_bootstrap(df, outcome_col=v)

    agg = df.groupby(["Agent", "Homeless"])[v].mean().reset_index()
    wide = agg.pivot(index="Agent", columns="Homeless", values=v)
    wide.columns = ["stable", "homeless"]
    wide = wide.dropna()
    diff = wide["homeless"] - wide["stable"]

    d = diff.mean() / (diff.std(ddof=1) + 1e-8)

    results.append({
        "Variable": v,
        "Difference": result["estimate"],
        "CI_lower": result["ci_lower"],
        "CI_upper": result["ci_upper"],
        "Cohens_d": d
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Cohens_d", ascending=False)

print(results_df)

           Variable  Difference  CI_lower  CI_upper  Cohens_d
1       Public_prop    0.540000  0.521423  0.558708  3.639344
9        HCV_active    0.174107  0.125000  0.227679  0.420612
0        Withdrawal    0.633929  0.348214  0.901786  0.295257
5  Partner_HCV_prop    0.011071  0.004194  0.018483  0.204013
3          Partners    0.093750  0.035714  0.160714  0.188720
2    Recept_sharing    0.151786 -0.098326  0.419754  0.077890
4          Drug_out   -0.031250 -0.196540  0.116183 -0.024846
7        Injections   -0.178571 -0.486607  0.156250 -0.067026
8         Alt_route   -1.763393 -2.236719 -1.321429 -0.530230
6         Treatment   -0.352679 -0.419643 -0.294643 -0.736475


In [ ]:
results_df.to_csv("outputs/claude_results_table.csv", index=False, encoding="utf-8")

In [9]:
table = pd.crosstab(df["Neighborhood"], df["Homeless"])
print(pd.crosstab(df["Neighborhood"], df["Homeless"], normalize="columns")) 

Homeless             0         1
Neighborhood                    
central       0.000000  0.035714
north side    0.361607  0.000000
south side    0.397321  0.120536
west side     0.241071  0.843750


In [11]:
table = pd.crosstab(df["Race"], df["Neighborhood"], normalize="index")
print(table)

Neighborhood   central  north side  south side  west side
Race                                                     
Hispanic      0.000000    0.017857    0.312500   0.669643
NHBlack       0.000000    0.000000    0.723214   0.276786
NHWhite       0.008929    0.294643    0.000000   0.696429
Other         0.062500    0.410714    0.000000   0.526786


In [22]:
def safe_corr(x, y):
    df_xy = pd.concat([x, y], axis=1).dropna()

    x_clean = df_xy.iloc[:, 0]
    y_clean = df_xy.iloc[:, 1]

    if x_clean.nunique() < 2 or y_clean.nunique() < 2:
        return np.nan
    return x_clean.corr(y_clean)


results = []
for a, b in itertools.combinations(measures, 2):

    corr = safe_corr(df[a], df[b])
    results.append({
        "var1": a,
        "var2": b,
        "correlation": corr
    })

corr_df = pd.DataFrame(results)
threshold = 0.3

strong_corr = corr_df[(corr_df["correlation"].abs() >= threshold)
    ].sort_values(by="correlation", key=lambda s: s.abs(), ascending=False)

strong_corr = strong_corr.reset_index(drop=True)

print(strong_corr)

                var1              var2  correlation
0     Recept_sharing          Drug_out     0.841737
1         Withdrawal        Injections     0.735150
2         Withdrawal    Recept_sharing     0.669558
3   Partner_HCV_prop        HCV_active     0.615511
4     Recept_sharing        Injections     0.564324
5         Withdrawal          Drug_out     0.563277
6           Partners        HCV_active     0.553770
7     Recept_sharing        HCV_active     0.516574
8           Drug_out        Injections     0.512383
9     Recept_sharing          Partners     0.479965
10          Drug_out        HCV_active     0.464275
11       Public_prop         Treatment    -0.443415
12        Withdrawal        HCV_active     0.436317
13    Recept_sharing         Alt_route     0.406864
14          Partners          Drug_out     0.403939
15        Withdrawal          Partners     0.399639
16        Injections         Alt_route     0.386999
17         Treatment         Alt_route     0.379780
18        In

In [ ]:
#cohens d function
def cohens_d(x0, x1):
    x0 = x0.dropna()
    x1 = x1.dropna()

    if len(x0) < 2 or len(x1) < 2:
        return np.nan

    n0, n1 = len(x0), len(x1)
    s0, s1 = x0.std(ddof=1), x1.std(ddof=1)

    pooled = np.sqrt(((n0 - 1)*s0**2 + (n1 - 1)*s1**2) / (n0 + n1 - 2))

    if pooled == 0 or np.isnan(pooled):
        return np.nan

    return (x1.mean() - x0.mean()) / pooled


In [28]:
baseline_vars = {
    "prior_injection": "Prev_Injection",
    "prior_sharing": "Prev_recept_sharing",
    "prior_network": "Prev_friend_entry",
}
output_vars = {
    "prior_injection": "Injections",
    "prior_sharing": "Recept_sharing",
    "prior_network": "Partner_HCV_prop",
}

results = []

for key in baseline_vars:
    base = baseline_vars[key]
    out = output_vars[key]

    sub = df[[base, out, "Homeless"]].dropna().copy()
    sub["change"] = sub[out] - sub[base]

    stable = sub[sub["Homeless"] == 0]["change"]
    homeless = sub[sub["Homeless"] == 1]["change"]

    d = cohens_d(stable, homeless)

    results.append({
        "relationship": f"{base} → {out}",
        "std_mean_change_effect": d
    })

result = pd.DataFrame(results)
print(result)

                           relationship  std_mean_change_effect
0           Prev_Injection → Injections               -0.160731
1  Prev_recept_sharing → Recept_sharing                0.016584
2  Prev_friend_entry → Partner_HCV_prop                0.133298


In [29]:
df = df.copy()
df["experience"] = df["Age"] - df["Age_Started"]
df["experience_group"] = np.where(df["experience"] >= 3, "Experienced", "Not experienced")
results = []

for y in measures:
    exp0 = df[df["experience_group"] == "Not experienced"]
    exp1 = df[df["experience_group"] == "Experienced"]

    d_not_exp = cohens_d(
        exp0[exp0["Homeless"] == 0][y],
        exp0[exp0["Homeless"] == 1][y]
    )
    d_exp = cohens_d(
        exp1[exp1["Homeless"] == 0][y],
        exp1[exp1["Homeless"] == 1][y]
    )
    results.append({
        "Outcome": y,
        "Difference (Experienced - Not)": d_exp - d_not_exp
    })

experience_results = pd.DataFrame(results).round(4)
print(experience_results)

            Outcome  Difference (Experienced - Not)
0        Withdrawal                         -0.1117
1       Public_prop                         -1.7825
2    Recept_sharing                          0.0436
3          Partners                          0.0446
4          Drug_out                          0.0120
5  Partner_HCV_prop                         -0.0028
6         Treatment                          0.2945
7        Injections                          0.0277
8         Alt_route                          0.0580
9        HCV_active                          0.0762


In [31]:
df["age_group"] = np.where(df["Age"] >= df["Age"].median(), "Old", "Young")
results = []

for y in measures:
    young_stable = df[(df["age_group"] == "Young") & (df["Homeless"] == 0)][y].dropna()
    young_homeless = df[(df["age_group"] == "Young") & (df["Homeless"] == 1)][y].dropna()

    old_stable = df[(df["age_group"] == "Old") & (df["Homeless"] == 0)][y]
    old_homeless = df[(df["age_group"] == "Old") & (df["Homeless"] == 1)][y]

    young_d = cohens_d(young_stable, young_homeless)
    old_d = cohens_d(old_stable, old_homeless)

    results.append({
        "Outcome": y,
        "Difference (Old - Young)": old_d - young_d if pd.notna(old_d) and pd.notna(young_d) else np.nan
    })

age_results = pd.DataFrame(results).round(4)
print(age_results)

            Outcome  Difference (Old - Young)
0        Withdrawal                   -0.2590
1       Public_prop                    0.7280
2    Recept_sharing                   -0.0574
3          Partners                    0.0104
4          Drug_out                   -0.0256
5  Partner_HCV_prop                   -0.0263
6         Treatment                   -0.0754
7        Injections                   -0.0166
8         Alt_route                    0.2627
9        HCV_active                   -0.0787


In [32]:
results = []
for y in measures:
    sub = df[["Gender", "Homeless", y]].dropna()

    female_stable = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 0)][y]
    female_homeless = sub[(sub["Gender"] == "Female") & (sub["Homeless"] == 1)][y]

    male_stable = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 0)][y]
    male_homeless = sub[(sub["Gender"] == "Male") & (sub["Homeless"] == 1)][y]

    female_d = cohens_d(female_stable, female_homeless)
    male_d = cohens_d(male_stable, male_homeless)

    results.append({
        "Outcome": y,
        "Difference (Female - Male)": female_d - male_d if pd.notna(female_d) and pd.notna(male_d) else np.nan
    })

gender_results = pd.DataFrame(results).round(4)
print(gender_results)

            Outcome  Difference (Female - Male)
0        Withdrawal                      0.3270
1       Public_prop                     -1.8917
2    Recept_sharing                     -0.0165
3          Partners                     -0.0138
4          Drug_out                     -0.0029
5  Partner_HCV_prop                     -0.0253
6         Treatment                      0.1012
7        Injections                      0.0324
8         Alt_route                      0.1986
9        HCV_active                     -0.0726


In [33]:
results = []

for y in measures:
    for race, g in df.groupby("Race"):
        stable = g[g["Homeless"] == 0][y]
        homeless = g[g["Homeless"] == 1][y]
        d = cohens_d(stable, homeless)

        results.append({
            "Outcome": y,
            "Race": race,
            "Cohens_d": d
        })
race_d_df = pd.DataFrame(results)

race_summary = race_d_df.groupby("Outcome")["Cohens_d"].agg(
    race_range=lambda x: x.max() - x.min(),
    mean="mean"
).reset_index()

print(race_summary)

            Outcome  race_range      mean
0         Alt_route    0.492507 -0.604623
1          Drug_out    0.037363  0.000237
2        HCV_active    0.190351  0.384053
3        Injections    0.033409 -0.006928
4  Partner_HCV_prop    0.029852  0.043696
5          Partners    0.140862  0.060727
6       Public_prop    3.740005  5.417681
7    Recept_sharing    0.045124  0.013925
8         Treatment    0.533815 -0.972604
9        Withdrawal    0.095682  0.231084
